In [0]:
--SELECT version();
--USE DATABASE qdp_database;

CREATE SCHEMA IF NOT EXISTS qdp_patients;
SET search_path TO qdp_patients;


-- metadata table
DROP TABLE IF EXISTS metadata;


CREATE TABLE metadata (
	supplier VARCHAR,
	data_product VARCHAR,
	data_product_version VARCHAR,
	schema_name VARCHAR,
	schema_version VARCHAR,
	schema_variant_name VARCHAR,
	schema_variant_version VARCHAR,
	instance VARCHAR,
	instance_version VARCHAR
 );



INSERT INTO metadata
(
  supplier,
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
)
VALUES (
  'quantexa.com',
  'com.quantexa.qdp_patients_all',
  '0.1',
  'patients.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);




-- -----------------------------------------------------------
-- create hub_patient table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS hub_patient CASCADE;

CREATE TABLE hub_patient (
    patient_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    patient_entity_id VARCHAR,
    person_entity_id VARCHAR,
  	tennant_id BIGINT NOT NULL DEFAULT 0,
	nhs_number VARCHAR,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE hub_patient ADD CONSTRAINT pk_patient_hubpatient_patientid PRIMARY KEY (patient_id);

COMMENT ON COLUMN hub_patient.patient_id IS 'Patient Identity';
COMMENT ON COLUMN hub_patient.patient_entity_id IS 'The unique Patient Composite Key identifier, generated during Quantexa Processing. May comprise multiple source NHS Numbers for merged source patient records';
COMMENT ON COLUMN hub_patient.nhs_number IS 'The unique Patient NHS Number identifier. See https://digital.nhs.uk/services/nhs-number';




-- -----------------------------------------------------------
-- create sat_patient table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_patient;

CREATE TABLE sat_patient (
    sat_patient_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    patient_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    record_source_id BIGINT NOT NULL DEFAULT 1,
	data_source_code VARCHAR,
	data_source_concept_id BIGINT,
	data_source_raw_code VARCHAR,
	data_source_raw_concept_id BIGINT,
	-- Cross-schema reference to qdp_persons.hub_person.person_id
	person_id BIGINT,
	active BOOLEAN DEFAULT TRUE,
--	birth_date DATE,
--	birth_datetime TIMESTAMP,
--	birth_date_numeric INT,
--	estimated_birth_date BOOLEAN DEFAULT FALSE,
--	estimated_birth_date_numeric INT,
--	is_multiple_birth BOOLEAN DEFAULT FALSE,
--	multiple_birth_number INT,
--	birth_place_city VARCHAR,
--	birth_place_district VARCHAR,
--	birth_place_country VARCHAR,
--	birth_place_country_code VARCHAR,
--	photo BYTEA,
--	is_deceased BOOLEAN DEFAULT FALSE,
--	deceased_datetime TIMESTAMP,
--	deceased_date_numeric INT,
--	gender_code VARCHAR,
--	gender_concept_id BIGINT,
--	gender_raw_code VARCHAR,
--	gender_raw_concept_id BIGINT,
--	marital_status_code VARCHAR,
--	marital_status_concept_id BIGINT,
--	marital_status_raw_code VARCHAR,
--	marital_status_raw_concept_id BIGINT,
--	ethnicity_code VARCHAR,
--	ethnicity_concept_id BIGINT,
--	ethnicity_raw_code VARCHAR,
--	ethnicity_raw_concept_id BIGINT,
--	religion_code VARCHAR,
--	religion_concept_id BIGINT,
--	religion_raw_code VARCHAR,
--	religion_raw_concept_id BIGINT,
--	nationality_code VARCHAR,
--	nationality_concept_id BIGINT,
--	nationality_raw_code VARCHAR,
--	nationality_raw_concept_id BIGINT,
--	country_of_residence_code VARCHAR,
--	country_of_residence_concept_id BIGINT,
--	country_of_residence_raw_code VARCHAR,
--	country_of_residence_raw_concept_id BIGINT,
--	is_interpreter_required BOOLEAN DEFAULT FALSE,
    primary_pratice_id VARCHAR,
	managing_organisation_id BIGINT,
	primary_address_id BIGINT,
	record_id BIGINT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_patient ADD CONSTRAINT pk_patient_satpatient_satpatientid PRIMARY KEY (sat_patient_id);
ALTER TABLE sat_patient ADD CONSTRAINT fk_patient_satpatient_hubpatient_patientid FOREIGN KEY (patient_id) REFERENCES hub_patient(patient_id);

COMMENT ON COLUMN sat_patient.patient_id IS 'Patient Identity';
COMMENT ON COLUMN sat_patient.sat_patient_id IS 'Patient Identity Sat';
COMMENT ON COLUMN sat_patient.person_id IS 'Cross-schema reference to qdp_persons_all.hub_person.person_id. Links the FHIR Patient to the canonical Person Entity';
COMMENT ON COLUMN sat_patient.active IS 'Whether this patient record is in active use. Many systems use this to mark records as non-current patients, such as those that have not been seen for a period of time based on an organisation''s business rules. See https://build.fhir.org/patient-definitions.html#Patient.active';
--COMMENT ON COLUMN sat_patient.birth_date IS 'The date of birth for the individual. See https://build.fhir.org/patient-definitions.html#Patient.birthDate';
--COMMENT ON COLUMN sat_patient.birth_datetime IS 'The date and time the person was born';
--COMMENT ON COLUMN sat_patient.birth_date_numeric IS 'Birth date as an integer numeric for link to Date Dimension for reporting purposes';
--COMMENT ON COLUMN sat_patient.estimated_birth_date IS 'True if the date and time of the person''s birth is an estimate not supported by any legal documentation';
--COMMENT ON COLUMN sat_patient.estimated_birth_date_numeric IS 'Estimated birth date as an integer numeric for link to Date Dimension for reporting purposes';
--COMMENT ON COLUMN sat_patient.is_multiple_birth IS 'Indicates whether the patient is part of a multiple birth. See https://build.fhir.org/patient-definitions.html#Patient.multipleBirth_x_';
--COMMENT ON COLUMN sat_patient.multiple_birth_number IS 'The number of siblings born at the same time (e.g. twin = 2, triplet = 3)';
--COMMENT ON COLUMN sat_patient.birth_place_city IS 'The city where this person was born';
--COMMENT ON COLUMN sat_patient.birth_place_district IS 'The district within a city where this person was born';
--COMMENT ON COLUMN sat_patient.birth_place_country IS 'The country where this person was born';
--COMMENT ON COLUMN sat_patient.birth_place_country_code IS 'The ISO 3166-1 Alpha-2 with extensions country code where this person was born (e.g. GB, GB-ENG, GB-SCT, GB-WLS)';
--COMMENT ON COLUMN sat_patient.photo IS 'Image of the patient. See https://build.fhir.org/patient-definitions.html#Patient.photo';
--COMMENT ON COLUMN sat_patient.is_deceased IS 'Indicates if the individual is deceased. See https://build.fhir.org/patient-definitions.html#Patient.deceased_x_';
--COMMENT ON COLUMN sat_patient.deceased_datetime IS 'The date and time the patient died';
--COMMENT ON COLUMN sat_patient.deceased_date_numeric IS 'Deceased date as an integer numeric for link to Date Dimension for reporting purposes';
--COMMENT ON COLUMN sat_patient.gender_code IS 'Administrative gender. E.g. male, female, other, unknown. See https://hl7.org/fhir/R5/valueset-administrative-gender.html';
--COMMENT ON COLUMN sat_patient.marital_status_code IS 'Most recent marital (civil) status of the patient. E.g. A (annulled), D (divorced), M (married), S (single), W (widowed), U (unknown). See https://hl7.org/fhir/R5/valueset-marital-status.html';
--COMMENT ON COLUMN sat_patient.ethnicity_code IS 'The administrative ethnicity code for the patient';
--COMMENT ON COLUMN sat_patient.religion_code IS 'The administrative religion code for the patient';
--COMMENT ON COLUMN sat_patient.nationality_code IS 'The nationality code for the patient';
--COMMENT ON COLUMN sat_patient.country_of_residence_code IS 'The ISO 3166-1 Alpha-2 with extensions country code of the patient''s current country of residence';
--COMMENT ON COLUMN sat_patient.is_interpreter_required IS 'True if an interpreter is required in order to communicate with this patient';
COMMENT ON COLUMN sat_patient.primary_pratice_id IS 'Primary Pratice Identifier that this Patient is register with and the custodian of the Patient Primary Care record.';
COMMENT ON COLUMN sat_patient.managing_organisation_id IS 'Organization that is the custodian of the patient record. Cross-schema reference to Organisation. See https://build.fhir.org/patient-definitions.html#Patient.managingOrganization';
COMMENT ON COLUMN sat_patient.primary_address_id IS 'Reference to the primary registered address of this patient. Cross-schema reference to qdp_locations_all.hub_address.address_id';
COMMENT ON COLUMN sat_patient.record_id IS 'A Unique Identifier for a Data Record within the System that allows Data Ownership, Ownership History, Dynamic Attributes and Metadata to be associated with that Record';


/**
-- -----------------------------------------------------------
-- create sat_patient_name table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_patient_name CASCADE;

CREATE TABLE sat_patient_name (
	patient_name_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	patient_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	rank INT,
	is_trusted BOOLEAN NOT NULL DEFAULT FALSE,
	is_preferred BOOLEAN NOT NULL DEFAULT FALSE,
	use_code VARCHAR,
	use_concept_id BIGINT,
	use_raw_code VARCHAR,
	use_raw_concept_id BIGINT,
	text VARCHAR,
	prefix VARCHAR,
	given1 VARCHAR,
	given1_soundex VARCHAR,
	given2 VARCHAR,
	given2_soundex VARCHAR,
	given3 VARCHAR,
	given3_soundex VARCHAR,
	family VARCHAR,
	family_soundex VARCHAR,
	suffix VARCHAR,
	full_name VARCHAR,
	full_name_soundex VARCHAR,
	full_name_raw VARCHAR,
	full_name_raw_soundex VARCHAR,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_patient_name ADD CONSTRAINT pk_patient_satpatientname_patientnameid PRIMARY KEY (patient_name_id);
ALTER TABLE sat_patient_name ADD CONSTRAINT fk_patient_satpatientname_hubpatient_patientid FOREIGN KEY (patient_id) REFERENCES hub_patient(patient_id);

COMMENT ON COLUMN sat_patient_name.patient_id IS 'Patient Identity';
COMMENT ON COLUMN sat_patient_name.patient_name_id IS 'Patient Name Identity';
COMMENT ON COLUMN sat_patient_name.rank IS 'The recommended order the patient names should be presented';
COMMENT ON COLUMN sat_patient_name.is_trusted IS 'Indicates whether this name has been verified as the trusted (golden record) version';
COMMENT ON COLUMN sat_patient_name.is_preferred IS 'Indicates whether this is the patient''s preferred name';
COMMENT ON COLUMN sat_patient_name.use_code IS 'The use of this name. E.g. usual, official, temp, nickname, anonymous, old, maiden. See https://hl7.org/fhir/R5/valueset-name-use.html';
COMMENT ON COLUMN sat_patient_name.text IS 'Text representation of the full name. See https://build.fhir.org/datatypes-definitions.html#HumanName.text';
COMMENT ON COLUMN sat_patient_name.prefix IS 'Parts that come before the name (e.g. Dr, Mr, Mrs, Prof). See https://build.fhir.org/datatypes-definitions.html#HumanName.prefix';
COMMENT ON COLUMN sat_patient_name.given1 IS 'First given name (forename). See https://build.fhir.org/datatypes-definitions.html#HumanName.given';
COMMENT ON COLUMN sat_patient_name.given1_soundex IS 'Soundex phonetic encoding of the first given name for fuzzy matching';
COMMENT ON COLUMN sat_patient_name.given2 IS 'Second given name (middle name)';
COMMENT ON COLUMN sat_patient_name.given2_soundex IS 'Soundex phonetic encoding of the second given name';
COMMENT ON COLUMN sat_patient_name.given3 IS 'Third given name';
COMMENT ON COLUMN sat_patient_name.given3_soundex IS 'Soundex phonetic encoding of the third given name';
COMMENT ON COLUMN sat_patient_name.family IS 'Family name (surname). See https://build.fhir.org/datatypes-definitions.html#HumanName.family';
COMMENT ON COLUMN sat_patient_name.family_soundex IS 'Soundex phonetic encoding of the family name for fuzzy matching';
COMMENT ON COLUMN sat_patient_name.suffix IS 'Parts that come after the name (e.g. Jr, III, PhD, OBE). See https://build.fhir.org/datatypes-definitions.html#HumanName.suffix';
COMMENT ON COLUMN sat_patient_name.full_name IS 'Assembled full name (given + family)';
COMMENT ON COLUMN sat_patient_name.full_name_soundex IS 'Soundex phonetic encoding of the full name';
COMMENT ON COLUMN sat_patient_name.full_name_raw IS 'The original full name as captured from the source system';
COMMENT ON COLUMN sat_patient_name.full_name_raw_soundex IS 'Soundex phonetic encoding of the raw full name';



-- -----------------------------------------------------------
-- create sat_patient_identifier table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_patient_identifier;

CREATE TABLE sat_patient_identifier (
	patient_identifier_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	patient_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	type_code VARCHAR,
	type_concept_id BIGINT,
	type_raw_code VARCHAR,
	type_raw_concept_id BIGINT,
	system VARCHAR,
	value VARCHAR,
	assigner_organisation_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_patient_identifier ADD CONSTRAINT pk_patient_satpatientidentifier_patientidentifierid PRIMARY KEY (patient_identifier_id);
ALTER TABLE sat_patient_identifier ADD CONSTRAINT fk_patient_satpatientidentifier_hubpatient_patientid FOREIGN KEY (patient_id) REFERENCES hub_patient(patient_id);

COMMENT ON COLUMN sat_patient_identifier.patient_id IS 'Patient Identity';
COMMENT ON COLUMN sat_patient_identifier.patient_identifier_id IS 'Patient Identifier Identity';
COMMENT ON COLUMN sat_patient_identifier.type_code IS 'A coded type for the identifier. E.g. MR (medical record number), NH (NHS Number), NI (National Insurance Number), PPN (passport number), DL (driving license number). See https://hl7.org/fhir/R5/valueset-identifier-type.html';
COMMENT ON COLUMN sat_patient_identifier.system IS 'The namespace for the identifier value. Establishes the context in which the value is unique. See https://build.fhir.org/datatypes-definitions.html#Identifier.system';
COMMENT ON COLUMN sat_patient_identifier.value IS 'The value that is unique within the system namespace. See https://build.fhir.org/datatypes-definitions.html#Identifier.value';
COMMENT ON COLUMN sat_patient_identifier.assigner_organisation_id IS 'Organization that issued/manages the identifier. See https://build.fhir.org/datatypes-definitions.html#Identifier.assigner';
COMMENT ON COLUMN sat_patient_identifier.rank IS 'The recommended order the Patient Identifier entries should be presented';



-- -----------------------------------------------------------
-- create sat_patient_telecom table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_patient_telecom;

CREATE TABLE sat_patient_telecom (
	patient_telecom_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	patient_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	system_code VARCHAR,
	system_concept_id BIGINT,
	system_raw_code VARCHAR,
	system_raw_concept_id BIGINT,
	value VARCHAR,
	value_display VARCHAR,
	value_raw VARCHAR,
	use_code VARCHAR,
	use_concept_id BIGINT,
	use_raw_code VARCHAR,
	use_raw_concept_id BIGINT,
	telephony_country_code VARCHAR,
	telephony_country_concept_id BIGINT,
	telephony_country_raw_code VARCHAR,
	telephony_country_raw_concept_id BIGINT,
	is_primary BOOLEAN DEFAULT FALSE,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_patient_telecom ADD CONSTRAINT pk_patient_satpatienttelecom_patienttelecomid PRIMARY KEY (patient_telecom_id);
ALTER TABLE sat_patient_telecom ADD CONSTRAINT fk_patient_satpatienttelecom_hubpatient_patientid FOREIGN KEY (patient_id) REFERENCES hub_patient(patient_id);

COMMENT ON COLUMN sat_patient_telecom.patient_id IS 'Patient Identity';
COMMENT ON COLUMN sat_patient_telecom.patient_telecom_id IS 'Patient Telecom Identity';
COMMENT ON COLUMN sat_patient_telecom.system_code IS 'Telecommunications form for contact point. E.g. phone, fax, email, pager, url, sms, other. See https://hl7.org/fhir/R5/valueset-contact-point-system.html';
COMMENT ON COLUMN sat_patient_telecom.value IS 'The actual contact point details (phone number, email address, etc.). See https://build.fhir.org/datatypes-definitions.html#ContactPoint.value';
COMMENT ON COLUMN sat_patient_telecom.value_display IS 'The formatted display version of the contact point value';
COMMENT ON COLUMN sat_patient_telecom.value_raw IS 'The original contact point value as captured from the source system';
COMMENT ON COLUMN sat_patient_telecom.use_code IS 'Purpose of this contact point. E.g. home, work, temp, old, mobile. See https://hl7.org/fhir/R5/valueset-contact-point-use.html';
COMMENT ON COLUMN sat_patient_telecom.telephony_country_code IS 'The international telephony country code (e.g. +44 for UK, +1 for US)';
COMMENT ON COLUMN sat_patient_telecom.is_primary IS 'Indicates whether this is the primary contact point for the patient';
COMMENT ON COLUMN sat_patient_telecom.rank IS 'Specifies a preferred order in which to use a set of contacts. ContactPoints with lower rank values are more preferred than those with higher rank values. See https://build.fhir.org/datatypes-definitions.html#ContactPoint.rank';



-- -----------------------------------------------------------
-- create sat_patient_contact table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_patient_contact;

CREATE TABLE sat_patient_contact (
	patient_contact_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	patient_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	relationship_code VARCHAR,
	relationship_concept_id BIGINT,
	relationship_raw_code VARCHAR,
	relationship_raw_concept_id BIGINT,
	contact_party_id BIGINT,
	contact_name VARCHAR,
	contact_given_name VARCHAR,
	contact_family_name VARCHAR,
	contact_gender_code VARCHAR,
	contact_gender_concept_id BIGINT,
	contact_gender_raw_code VARCHAR,
	contact_gender_raw_concept_id BIGINT,
	contact_telecom_value VARCHAR,
	contact_telecom_system_code VARCHAR,
	contact_telecom_use_code VARCHAR,
	contact_organisation_id BIGINT,
	is_guardian BOOLEAN NOT NULL DEFAULT FALSE,
	is_primary BOOLEAN NOT NULL DEFAULT FALSE,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_patient_contact ADD CONSTRAINT pk_patient_satpatientcontact_patientcontactid PRIMARY KEY (patient_contact_id);
ALTER TABLE sat_patient_contact ADD CONSTRAINT fk_patient_satpatientcontact_hubpatient_patientid FOREIGN KEY (patient_id) REFERENCES hub_patient(patient_id);

COMMENT ON COLUMN sat_patient_contact.patient_id IS 'Patient Identity';
COMMENT ON COLUMN sat_patient_contact.patient_contact_id IS 'Patient Contact Identity';
COMMENT ON COLUMN sat_patient_contact.relationship_code IS 'The nature of the relationship between the patient and the contact person. E.g. C (emergency contact), E (employer), F (federal agency), I (insurance company), N (next-of-kin), S (state agency), U (unknown). See https://hl7.org/fhir/R5/valueset-patient-contactrelationship.html';
COMMENT ON COLUMN sat_patient_contact.contact_party_id IS 'The resolved Party Identity of the contact person, if known';
COMMENT ON COLUMN sat_patient_contact.contact_name IS 'A name associated with the contact person. See https://build.fhir.org/patient-definitions.html#Patient.contact.name';
COMMENT ON COLUMN sat_patient_contact.contact_given_name IS 'First given name of the contact person';
COMMENT ON COLUMN sat_patient_contact.contact_family_name IS 'Family name of the contact person';
COMMENT ON COLUMN sat_patient_contact.contact_gender_code IS 'Administrative gender of the contact person. See https://hl7.org/fhir/R5/valueset-administrative-gender.html';
COMMENT ON COLUMN sat_patient_contact.contact_telecom_value IS 'Primary contact detail for the contact person (phone number, email, etc.)';
COMMENT ON COLUMN sat_patient_contact.contact_telecom_system_code IS 'Telecommunications system for the contact (phone, email, etc.)';
COMMENT ON COLUMN sat_patient_contact.contact_telecom_use_code IS 'Purpose of the contact detail (home, work, etc.)';
COMMENT ON COLUMN sat_patient_contact.contact_organisation_id IS 'Organization that is associated with the contact. See https://build.fhir.org/patient-definitions.html#Patient.contact.organization';
COMMENT ON COLUMN sat_patient_contact.is_guardian IS 'Indicates whether this contact person is a legal guardian';
COMMENT ON COLUMN sat_patient_contact.is_primary IS 'Indicates whether this is the primary contact for the patient';
COMMENT ON COLUMN sat_patient_contact.rank IS 'The recommended order the Patient Contact entries should be presented';



-- -----------------------------------------------------------
-- create sat_patient_communication table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_patient_communication;

CREATE TABLE sat_patient_communication (
	patient_communication_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	patient_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	language_code VARCHAR,
	language_concept_id BIGINT,
	language_raw_code VARCHAR,
	language_raw_concept_id BIGINT,
	preferred BOOLEAN NOT NULL DEFAULT FALSE,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_patient_communication ADD CONSTRAINT pk_patient_satpatientcomm_patientcommid PRIMARY KEY (patient_communication_id);
ALTER TABLE sat_patient_communication ADD CONSTRAINT fk_patient_satpatientcomm_hubpatient_patientid FOREIGN KEY (patient_id) REFERENCES hub_patient(patient_id);

COMMENT ON COLUMN sat_patient_communication.patient_id IS 'Patient Identity';
COMMENT ON COLUMN sat_patient_communication.patient_communication_id IS 'Patient Communication Identity';
COMMENT ON COLUMN sat_patient_communication.language_code IS 'A language which may be used to communicate with the patient about their health. BCP-47 language tags (e.g. en, en-GB, cy, fr). See https://build.fhir.org/patient-definitions.html#Patient.communication.language';
COMMENT ON COLUMN sat_patient_communication.preferred IS 'Indicates whether this language is the patient''s preferred language for communication. See https://build.fhir.org/patient-definitions.html#Patient.communication.preferred';
COMMENT ON COLUMN sat_patient_communication.rank IS 'The recommended order the Patient Communication entries should be presented';



-- -----------------------------------------------------------
-- create sat_patient_general_practitioner table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_patient_general_practitioner;

CREATE TABLE sat_patient_general_practitioner (
	patient_general_practitioner_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	patient_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	practitioner_party_id BIGINT,
	practitioner_role_code VARCHAR,
	practitioner_role_concept_id BIGINT,
	practitioner_role_raw_code VARCHAR,
	practitioner_role_raw_concept_id BIGINT,
	practice_organisation_id BIGINT,
	practice_ods_code VARCHAR,
	is_primary BOOLEAN NOT NULL DEFAULT FALSE,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_patient_general_practitioner ADD CONSTRAINT pk_patient_satpatientgp_patientgpid PRIMARY KEY (patient_general_practitioner_id);
ALTER TABLE sat_patient_general_practitioner ADD CONSTRAINT fk_patient_satpatientgp_hubpatient_patientid FOREIGN KEY (patient_id) REFERENCES hub_patient(patient_id);

COMMENT ON COLUMN sat_patient_general_practitioner.patient_id IS 'Patient Identity';
COMMENT ON COLUMN sat_patient_general_practitioner.patient_general_practitioner_id IS 'Patient General Practitioner Identity';
COMMENT ON COLUMN sat_patient_general_practitioner.practitioner_party_id IS 'The patient''s nominated primary care provider. E.g. Organization (GP practice), Practitioner (named GP), PractitionerRole. See https://build.fhir.org/patient-definitions.html#Patient.generalPractitioner';
COMMENT ON COLUMN sat_patient_general_practitioner.practitioner_role_code IS 'The role of the practitioner (e.g. general medical practitioner, practice nurse)';
COMMENT ON COLUMN sat_patient_general_practitioner.practice_organisation_id IS 'The GP practice organisation Identity. Cross-schema reference to Organisation';
COMMENT ON COLUMN sat_patient_general_practitioner.practice_ods_code IS 'The ODS (Organisation Data Service) code for the GP practice. See https://digital.nhs.uk/services/organisation-data-service';
COMMENT ON COLUMN sat_patient_general_practitioner.is_primary IS 'Indicates whether this is the patient''s primary GP registration';
COMMENT ON COLUMN sat_patient_general_practitioner.rank IS 'The recommended order the Patient General Practitioner entries should be presented';



-- -----------------------------------------------------------
-- create sat_patient_link table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_patient_link;

CREATE TABLE sat_patient_link (
	patient_link_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	patient_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	other_patient_id BIGINT,
	other_patient_entity_id VARCHAR,
	link_type_code VARCHAR,
	link_type_concept_id BIGINT,
	link_type_raw_code VARCHAR,
	link_type_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_patient_link ADD CONSTRAINT pk_patient_satpatientlink_patientlinkid PRIMARY KEY (patient_link_id);
ALTER TABLE sat_patient_link ADD CONSTRAINT fk_patient_satpatientlink_hubpatient_patientid FOREIGN KEY (patient_id) REFERENCES hub_patient(patient_id);

COMMENT ON COLUMN sat_patient_link.patient_id IS 'Patient Identity';
COMMENT ON COLUMN sat_patient_link.patient_link_id IS 'Patient Link Identity';
COMMENT ON COLUMN sat_patient_link.other_patient_id IS 'The other patient or related person resource that the link refers to. See https://build.fhir.org/patient-definitions.html#Patient.link.other';
COMMENT ON COLUMN sat_patient_link.other_patient_entity_id IS 'The entity ID of the linked patient (for Quantexa entity resolution context)';
COMMENT ON COLUMN sat_patient_link.link_type_code IS 'The type of link between this patient and another patient or related person resource. E.g. replaced-by (this patient record has been replaced), replaces (this patient record replaces another), refer (the patient resource should be consulted as an alternative), seealso (the patient resource is known to contain the same person). See https://hl7.org/fhir/R5/valueset-link-type.html';
COMMENT ON COLUMN sat_patient_link.rank IS 'The recommended order the Patient Link entries should be presented';


**/
CREATE OR REPLACE VIEW view_patients AS
SELECT
    h.tennant_id,
    t.tennant_name,
    h.patient_id,
    h.patient_entity_id,
    h.nhs_number,
    h.period_start AS hub_period_start,
    h.period_end AS hub_period_end,
    s.sat_patient_id,
    s.person_id,
    s.active,
    s.primary_pratice_id,
    s.managing_organisation_id,
    s.primary_address_id,
    s.record_id,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_patients.hub_patient h
    JOIN budd.qdp_patients.sat_patient s
      ON h.patient_id = s.patient_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_patients;

/**
-- -----------------------------------------------------------
-- create view_patients view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_patients AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.patient_id,
  h.patient_entity_id,
  h.nhs_number,
  h.period_start AS hub_period_start,
  h.period_end AS hub_period_end,
  s.sat_patient_id,
  s.individual_id,
  s.active,
  pn.given1 AS given_name,
  pn.family AS family_name,
  pn.full_name,
  s.birth_date,
  s.birth_datetime,
  s.birth_date_numeric,
  s.estimated_birth_date,
  s.is_multiple_birth,
  s.multiple_birth_number,
  s.birth_place_city,
  s.birth_place_district,
  s.birth_place_country,
  s.birth_place_country_code,
  s.is_deceased,
  s.deceased_datetime,
  s.deceased_date_numeric,
  s.gender_code,
  s.gender_concept_id,
  s.gender_raw_code,
  s.gender_raw_concept_id,
  s.marital_status_code,
  s.marital_status_concept_id,
  s.marital_status_raw_code,
  s.marital_status_raw_concept_id,
  s.ethnicity_code,
  s.ethnicity_concept_id,
  s.ethnicity_raw_code,
  s.ethnicity_raw_concept_id,
  s.religion_code,
  s.religion_concept_id,
  s.religion_raw_code,
  s.religion_raw_concept_id,
  s.nationality_code,
  s.nationality_concept_id,
  s.nationality_raw_code,
  s.nationality_raw_concept_id,
  s.country_of_residence_code,
  s.country_of_residence_concept_id,
  s.country_of_residence_raw_code,
  s.country_of_residence_raw_concept_id,
  s.is_interpreter_required,
  s.managing_organisation_id,
  s.primary_address_id,
  s.record_id,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_patients.hub_patient h
    JOIN budd.qdp_patients.sat_patient s
      ON h.patient_id = s.patient_id
    LEFT JOIN budd.qdp_patients.sat_patient_name pn
      ON h.patient_id = pn.patient_id AND pn.is_trusted = TRUE
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_patients;


-- -----------------------------------------------------------
-- create view_patient_names view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_patient_names AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.patient_id,
	h.patient_entity_id,
	h.nhs_number,
  s.gender_code,
  s.birth_date,
  s.is_deceased,
	pn.patient_name_id,
	pn.rank AS name_rank,
	pn.is_trusted AS is_trusted_name,
	pn.is_preferred AS is_preferred_name,
	pn.use_code AS name_use_code,
	pn.use_concept_id AS name_use_concept_id,
	pn.use_raw_code AS name_use_raw_code,
	pn.use_raw_concept_id AS name_use_raw_concept_id,
	pn.text AS name_text,
	pn.prefix AS name_prefix,
	pn.given1,
	pn.given1_soundex,
	pn.given2,
	pn.given2_soundex,
	pn.given3,
	pn.given3_soundex,
	pn.family,
	pn.family_soundex,
	pn.suffix AS name_suffix,
	pn.full_name,
	pn.full_name_soundex,
	pn.full_name_raw,
	pn.full_name_raw_soundex,
	pn.period_start AS name_period_start,
	pn.period_end AS name_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_patients.hub_patient h
    JOIN budd.qdp_patients.sat_patient s
      ON h.patient_id = s.patient_id
    JOIN budd.qdp_patients.sat_patient_name pn
      ON h.patient_id = pn.patient_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
    ORDER BY pn.rank
;

SELECT * from view_patient_names;


-- -----------------------------------------------------------
-- create view_patient_identifiers view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_patient_identifiers AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.patient_id,
	h.patient_entity_id,
	h.nhs_number,
  s.gender_code,
  s.birth_date,
  s.is_deceased,
	pi.patient_identifier_id,
	pi.type_code AS identifier_type_code,
	pi.type_concept_id AS identifier_type_concept_id,
	pi.type_raw_code AS identifier_type_raw_code,
	pi.type_raw_concept_id AS identifier_type_raw_concept_id,
	pi.system AS identifier_system,
	pi.value AS identifier_value,
	pi.assigner_organisation_id,
	pi.rank AS identifier_rank,
	pi.period_start AS identifier_period_start,
	pi.period_end AS identifier_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_patients.hub_patient h
    JOIN budd.qdp_patients.sat_patient s
      ON h.patient_id = s.patient_id
    JOIN budd.qdp_patients.sat_patient_identifier pi
      ON h.patient_id = pi.patient_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
    ORDER BY pi.rank
;

SELECT * from view_patient_identifiers;


-- -----------------------------------------------------------
-- create view_patient_contacts view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_patient_contacts AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.patient_id,
	h.patient_entity_id,
	h.nhs_number,
  s.gender_code,
  s.birth_date,
	pc.patient_contact_id,
	pc.relationship_code,
	pc.relationship_concept_id,
	pc.relationship_raw_code,
	pc.relationship_raw_concept_id,
	pc.contact_party_id,
	pc.contact_name,
	pc.contact_given_name,
	pc.contact_family_name,
	pc.contact_gender_code,
	pc.contact_telecom_value,
	pc.contact_telecom_system_code,
	pc.contact_telecom_use_code,
	pc.contact_organisation_id,
	pc.is_guardian,
	pc.is_primary AS is_primary_contact,
	pc.rank AS contact_rank,
	pc.period_start AS contact_period_start,
	pc.period_end AS contact_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_patients.hub_patient h
    JOIN budd.qdp_patients.sat_patient s
      ON h.patient_id = s.patient_id
    JOIN budd.qdp_patients.sat_patient_contact pc
      ON h.patient_id = pc.patient_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
    ORDER BY pc.rank
;

SELECT * from view_patient_contacts;


-- -----------------------------------------------------------
-- create view_patient_gp_registrations view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_patient_gp_registrations AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.patient_id,
	h.patient_entity_id,
	h.nhs_number,
  s.gender_code,
  s.birth_date,
  s.is_deceased,
	gp.patient_general_practitioner_id,
	gp.practitioner_party_id,
	gp.practitioner_role_code,
	gp.practitioner_role_concept_id,
	gp.practitioner_role_raw_code,
	gp.practitioner_role_raw_concept_id,
	gp.practice_organisation_id,
	gp.practice_ods_code,
	gp.is_primary AS is_primary_gp,
	gp.rank AS gp_rank,
	gp.period_start AS gp_period_start,
	gp.period_end AS gp_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_patients.hub_patient h
    JOIN budd.qdp_patients.sat_patient s
      ON h.patient_id = s.patient_id
    JOIN budd.qdp_patients.sat_patient_general_practitioner gp
      ON h.patient_id = gp.patient_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
    ORDER BY gp.rank
;

SELECT * from view_patient_gp_registrations;


**/
